<a href="https://colab.research.google.com/github/swapspingle/IK_Agent_programs/blob/main/CRM_Lead_Qualifier_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [3]:
try:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=userdata.get('OPENROUTER_API_KEY'),
    )
except Exception as e:
    print(f"Error initializing OpenAI client: {e}")
    print("Please ensure your OPENROUTER_APIKEY is set in Colab Secrets.")

In [4]:
def lookup_domain_info(domain: str) -> str:
    """
    Looks up and returns mock company information based on its domain.
    In a real system, this would call an external data enrichment API (e.g., Clearbit).
    """
    print(f"-> TOOL ACTIVATED: Looking up domain info for {domain}...")

    # Mock database for demonstration
    mock_data = {
        "acmecorp.com": {"industry": "Software/SaaS", "size": "501-1000 employees", "revenue": "$50M - $100M"},
        "widgetco.net": {"industry": "Manufacturing", "size": "100-250 employees", "revenue": "$10M - $25M"},
        "globalfin.org": {"industry": "Financial Services", "size": "5000+ employees", "revenue": "$1B+"},
    }

    info = mock_data.get(domain,{"industry": "Unknown", "size": "N/A", "revenue": "N/A"})

    return json.dumps(info)

In [5]:
def check_crm_history(email: str) -> str:
    """
    Checks the internal CRM system for past engagement history with the lead.
    In a real system, this would query a PostgreSQL database or a CRM API (e.g., Salesforce).
    """
    print(f"-> TOOL ACTIVATED: Checking CRM history for {email}...")

    # Mock database for demonstration
    mock_data = {
        "jane@acmecorp.com": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."},
        "bob@widgetco.net": {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
        "default": {"last_contact": "N/A", "status": "No Record", "notes": "New lead, first contact opportunity."},
    }

    history = mock_data.get(email, mock_data["default"])
    return json.dumps(history)

In [6]:
def calculate_lead_score(data_summary: str) -> str:
    """
    Analyzes the collected data (domain and CRM history) to assign a lead score (High/Medium/Low).
    This function simulates a complex scoring algorithm.
    """
    print("-> TOOL ACTIVATED: Calculating lead score...")

    data = json.loads(data_summary)
    score = "Low" # Default score

    # Simple scoring logic for demonstration
    if data["domain_info"].get("revenue", "").startswith("$1B+"):
        score = "High"
    elif data["crm_history"].get("status") == "Active Opportunity":
        score = "High"
    elif data["domain_info"].get("revenue", "").startswith("$50M"):
        score = "Medium"

    return json.dumps({"lead_score": score})

In [7]:
# Map of available function names to the actual Python functions
AVAILABLE_FUNCTIONS = {
    "lookup_domain_info": lookup_domain_info,
    "check_crm_history": check_crm_history,
    "calculate_lead_score": calculate_lead_score,
}

**Configuring the Agent's "Menu" (Tool Schema)**

The Large Language Model (LLM) cannot see our Python code directly. We must describe our tools to it using a specific JSON format known as a Schema.

This schema tells the model:

* What the tool does (Description).
* When to use it (Context).
* How to use it (Parameters/Arguments).


We pass this list to the tools parameter in the API call later.
It effectively gives the AI a "menu" of actions it can take.

In [8]:
# This is how the AI model learns about the tools it can use.
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "lookup_domain_info",
            "description": "Retrieves general business information (industry, size, revenue) about a company based on its domain name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "domain": {"type": "string", "description": "The company's domain name, e.g., 'acmecorp.com'"},
                },
                "required": ["domain"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_crm_history",
            "description": "Checks the internal CRM system for past contact, status, and notes associated with a specific lead email.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string", "description": "The full email address of the lead."},
                },
                "required": ["email"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_lead_score",
            "description": "Calculates the priority score (High/Medium/Low) for a lead based on a summary of all collected domain and CRM history data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data_summary": {"type": "string", "description": "A JSON string containing the combined domain_info and crm_history."},
                },
                "required": ["data_summary"],
            },
        },
    },
]

## The Agent Loop (Think → Act → Observe)

This is the brain of the application. The run_agent function implements the core feedback loop required for an autonomous agent.

**How it works:**

1.  **State Management (collected_data):** We initialize a dictionary outside the
loop to act as the agent's "memory" across multiple steps.
2. **Think**: We send the conversation history to the model (client.chat.completions.create).
3. **Act**: If the model decides to call a tool, we execute the corresponding Python function.
4. **Observe**: We append the tool's output to the conversation history as a new message.
5. **Repeat**: The loop continues until the model decides it has enough information to answer the user directly.

In [13]:
def run_agent(user_prompt: str):
    """
    The main execution loop for the CRM Lead Qualifier Agent.
    """
    print(f"\n--- Running Lead Qualifier Agent ---")

    system_prompt = (
        "You are an expert CRM Lead Qualifier Agent. Your sole task is to analyze a sales lead "
        "provided via email address. You must follow these steps precisely: "
        "1. Identify the domain from the email. "
        "2. Call `lookup_domain_info` and `check_crm_history` sequentially to gather all data. "
        "3. Combine all collected data into a single JSON object. "
        "4. Call `calculate_lead_score` with the combined JSON object. "
        "5. Finally, synthesize all information (domain info, CRM history, and score) "
        "into a single, easy-to-read summary for a busy sales rep."
    )

    messages=[
        {"role":"system", "content":system_prompt},
        {"role":"user", "content":user_prompt}
    ]

    collected_data = {}

    while True:
      print("\n[AI Thinking...]")

      response= client.chat.completions.create(
          model="openai/gpt-6-luna", # $0.10 in / $0.50 out per 1M tokens (gpt-4o was $2.50 / $10)
            # Other options: "openai/gpt-5.6-luna", "openai/gpt-5.4-nano", "anthropic/claude-haiku-5.5"
          messages=messages,
          tools=tools_schema,
          tool_choice="auto"
      )

      response_message = response.choices[0].message
      print(f"[AI Response]: {response_message}")
      messages.append(response_message)

      if(response_message.tool_calls):

        tool_calls = response_message.tool_calls

        for tool_call in tool_calls:

          function_name= tool_call.function.name
          function_to_call= AVAILABLE_FUNCTIONS.get(function_name)
          function_args= json.loads(tool_call.function.arguments)

          if not function_to_call:
            print(f"Error: Unknown function {function_name}")
            continue

          #Execute the function
          if(function_name=="calculate_lead_score"):
            function_args={"data_summary": json.dumps(collected_data)}

          function_result= function_to_call(**function_args)

          if(function_name=="lookup_domain_info"):
            collected_data["domain_info"]=json.loads(function_result)
          elif(function_name=="check_crm_history"):
            collected_data["crm_history"]=json.loads(function_result)

          # Append the tool result as a NEW message
          messages.append({
              "role": "tool",
              "tool_call_id": tool_call.id,
              "name": function_name,
              "content": function_result
          })
      else:
        print("\n__final summary__")
        print(f"\n{response_message.content}")
        break

      # print(f"Response {response_message}")
      # break

In [14]:
# Scenario 1: High-Value Lead (Large company, needs scoring)
lead_email_1 = "jane@acmecorp.com"
run_agent(f"Please qualify this lead for my call tomorrow: {lead_email_1}")

print("\n" + "="*80 + "\n")


--- Running Lead Qualifier Agent ---

[AI Thinking...]
[AI Response]: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_48a1eddd11824cb886544803ced6e843', function=Function(arguments='{"domain":"acmecorp.com"}', name='lookup_domain_info'), type='function', index=0)], reasoning=None, reasoning_details=[{'type': 'reasoning.encrypted', 'data': 'gAAAAABrAxV7vUx0z6nLCLvhHBC36Sa27piDdVH-ItDJjYVHfkRF_JgAy6cYL4iTx0-EdngyFupQyRGFqYfKimcAO85MN4ryjyGY-GR9MmCvNGzHW1rIaPMnhB6OyxhILa13OCqOoVatkAtR02NRF0svHDIfU_a4sA5kKXKr3YA7AR8j1Da42YtqhEj5qq8HC2XA-pjQaAXkrYg1f9yJS7AgW7Ni29xAOjqERs9OpJE7Dz8xEavi6iQb-LJbc8_QBOp8fGyZmXPpG2u_vlK90FBf6q1MOSfB6zJ0TsKeauAXPXtUvZo6q20wfhSjEjxR7aMEwC551L-qla7wNfdLSNVHMal5L6QalpQvX7j925RTUBjwMQrKShX2jHP25fg3SIbAJhpwfE3TwLySNEcskDKEd8xc-uj-X2E5fKFQ0lvMG2UpvqtNe3xaRsaZktH715hhONGWdMEm9WIHh1P4Px-ORxt1Ry8-Dyd2595fdSHQ5shFwhVc4Q30UlwoI56i1PeH2lS33XFCU_H7

In [15]:
lead_email2="bob@widgetco.net"
run_agent(f"Please qualify this lead for my call tomorrow:{lead_email2}")


--- Running Lead Qualifier Agent ---

[AI Thinking...]
[AI Response]: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_3a70bcd447864f668c014474ed970344', function=Function(arguments='{"domain":"widgetco.net"}', name='lookup_domain_info'), type='function', index=0)], reasoning='**Following workflow steps**\n\nI need to follow a sequential process for looking things up and using the CRM, rather than doing them in parallel. First, I should extract information from widgetco.net, then call the lookup function, followed by the CRM. It’s crucial to maintain this order to ensure everything works properly and efficiently. I want to make sure I’m doing things right to avoid any confusion later on.', reasoning_details=[{'type': 'reasoning.summary', 'summary': '**Following workflow steps**\n\nI need to follow a sequential process for looking things up and using the CRM, 